In [2]:
import re
from enum import Enum
from dataclasses import dataclass
from typing import Optional  # Toujours nécessaire pour Optional

# Équivalents des énumérations TypeScript
class Genre(Enum):
    HOMME = "HOMME"
    FEMME = "FEMME"
    # Ajoutez les autres valeurs selon votre besoin

class CodeDossier(Enum):
    TRA = "TRA"
    LIV = "LIV"
    # Ajoutez les autres valeurs selon votre besoin

class Phase(Enum):
    DUPLI = "DUPLI"
    PROD = "PROD"
    # Ajoutez les autres valeurs selon votre besoin

# Utilisation de dataclass pour PathInfos (Python 3.7+)
@dataclass
class PathInfos:
    trimestre: Optional[str] = None
    genre: Optional[Genre] = None
    collection: Optional[str] = None
    animation: Optional[str] = None
    code_dossier: Optional[CodeDossier] = None
    phase: Optional[Phase] = None
    segment: Optional[str] = None
    produit: Optional[str] = None
    dimensions: Optional[str] = None
    version: Optional[int] = None
    date: Optional[str] = None

def extract_basename_and_extension(filename: str) -> tuple[str, str]:
    """
    Extrait le nom de base et l'extension d'un nom de fichier.

    Args:
        filename: Le nom du fichier

    Returns:
        Un tuple contenant le nom de base et l'extension
    """
    parts = filename.split('.')
    if len(parts) == 1:
        return filename, ""
    extension = parts[-1]
    basename = '.'.join(parts[:-1])
    return basename, extension

def parse_path(path: str) -> tuple[PathInfos, bool, list[str]]:
    """
    Analyse un chemin de fichier et en extrait des informations structurées basées sur les conventions de nommage des dossiers/fichiers en vigueur.

    La fonction suppose que la chaîne de caractères représentant le chemin, et en particulier le nom du fichier, suit un format spécifique.
    Chaque segment du chemin est analysé et les informations sur le chemin sont renvoyées dans un objet structuré.

    Args:
        path: Le chemin de fichier à analyser.

    Returns:
        Un 3-uple contenant :
          - Un objet `PathInfos` rempli avec les détails extraits du chemin. Si certains éléments ne peuvent être trouvés,
            ils resteront nuls dans l'objet PathInfos.
          - Un booléen indiquant si le chemin de fichier est valide : il vaudra `False` si le chemin, et en particulier le nom de fichier,
            ne respecte pas le format attendu ; il vaudra `True` sinon.
          - Une liste de chaînes de caractères représentant les erreurs rencontrées lors de l'extraction des informations.
    """

    # On part d'un objet qui ne contient aucune information
    path_infos = PathInfos()

    # Chaîne vide
    if not path:
        return path_infos, False, ["Le chemin est vide."]

    # Les noms de dossiers sont séparés par le caractère `/` ; on gère le cas où plusieurs `/` se suivent
    parts = [part for part in path.split('/') if part]

    i = 0
    # On trouve l'index du premier dossier dont le nom ressemble à un trimestre (exemple : `24-Q1`)
    while i < len(parts) and not re.match(r'^\d{2}-Q\d$', parts[i]):
        i += 1

    # On s'arrête immédiatement si le chemin ne contient pas le trimestre
    if i == len(parts):
        return path_infos, False, []

    # On détermine l'index du dossier représentant le segment (qui doit être le dossier parent du fichier)
    last_part_index = i

    # Le premier dossier est donc nommé d'après le trimestre
    path_infos.trimestre = parts[i]

    # Le dossier suivant est nommé d'après le genre
    try:
        path_infos.genre = Genre[parts[i + 1]] if i + 1 < len(parts) else None
    except KeyError:
        path_infos.genre = None

    # Vérification si le chemin contient un dossier de travail ou une collection
    try:
        if i + 3 < len(parts) and parts[i + 3] in [e.name for e in CodeDossier]:  # Animation HORS d'une collection
            path_infos.animation = parts[i + 2] if i + 2 < len(parts) else None
            path_infos.code_dossier = CodeDossier[parts[i + 3]] if i + 3 < len(parts) else None

            # On vérifie si le chemin contient la phase après le code du dossier
            if i + 4 < len(parts) and parts[i + 4] in [e.name for e in Phase]:
                path_infos.phase = Phase[parts[i + 4]] if i + 4 < len(parts) else None
                path_infos.segment = parts[i + 5] if i + 5 < len(parts) else None
                last_part_index += 5
            else:
                path_infos.segment = parts[i + 4] if i + 4 < len(parts) else None
                last_part_index += 4
        else:  # Animation DANS une collection
            path_infos.collection = parts[i + 2] if i + 2 < len(parts) else None
            path_infos.animation = parts[i + 3] if i + 3 < len(parts) else None

            if i + 4 < len(parts) and parts[i + 4] in [e.name for e in CodeDossier]:
                path_infos.code_dossier = CodeDossier[parts[i + 4]]

            # On vérifie si le chemin contient la phase après le code du dossier
            if i + 5 < len(parts) and parts[i + 5] in [e.name for e in Phase]:
                path_infos.phase = Phase[parts[i + 5]] if i + 5 < len(parts) else None
                path_infos.segment = parts[i + 6] if i + 6 < len(parts) else None
                last_part_index += 6
            else:
                # Actuellement n'importe quelle chaîne de caractères est prise pour un segment
                path_infos.segment = parts[i + 5] if i + 5 < len(parts) else None
                last_part_index += 5
    except (IndexError, KeyError):
        pass

    # On vérifie que le dossier représentant le segment soit bien le dossier parent du fichier
    if last_part_index != len(parts) - 2:
        return path_infos, False, ["Le dossier parent ne représente pas le segment (ex : SLG, MARO, etc.)."]

    # La dernière partie du chemin est le nom du fichier
    filename = parts[len(parts) - 1]
    basename, _ = extract_basename_and_extension(filename)

    prefix = path_infos.trimestre + "-"
    if path_infos.collection:
        prefix += path_infos.collection + "-"

    # Le nom du fichier doit être correctement prefixé : TRIMESTRE-[COLLECTION-]ANIMATION-
    prefix += path_infos.animation + "-"

    if not basename.startswith(prefix):
        return path_infos, False, [f"Le nom du fichier doit commencer par : {prefix}"]

    # La partie restante doit suivre un format spécifique : NOM-PRODUIT-(VERSION)-DATE
    suffix = basename[len(prefix):]

    # Exemple : MOTIF-SMALL-500X500-(02)-250320
    regex = r'^([A-Z0-9-]+)(?:-(\d+x\d+))?-\((\d+)\)-(\d{6})$'
    match = re.match(regex, suffix)

    if not match:
        return path_infos, False, [f"La fin du nom du fichier n'a pas le format attendu : {suffix}"]

    produit, dimensions, version, date = match.groups()
    path_infos.produit = produit

    if dimensions:
        path_infos.produit += "-" + dimensions  # On considère que les dimensions font partie du nom du produit

    path_infos.dimensions = dimensions
    path_infos.version = int(version)
    path_infos.date = date

    return path_infos, True, []

In [7]:
test_paths = [
    ["/Volumes/Public/TRANSFORMATIONS/24-Q4/HOMME/MNG-TRANSPARENT/TRA/TRAVEL/24-Q4-MNG-TRANSPARENT-POCHETTE-TRIO-(03)-240108.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-TRAVEL-BAG-48H-LAGOON-(01)-231122.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-POCHETTE-COSMETIQUE-(05)-240208.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-POCHETTE-COSMETIQUE-(02)-231207.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-KEEPALL-45-(03)-240208.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-HORIZON-55-(02)-231220.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-ZIPPY-WALLET-FRAMBOISE-(01)-240104.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-ZIPPY-WALLET-(011)-231221.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-ZIPPY-COIN-PURSE-(04)-231221.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-PORTE-CARTES-SIMPLE-TURQUOISE-(04)-240111.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/LG/25-Q1-ANIMATION-SPEEDY-25-(01)-240923.ai", False, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/DUPLI/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/PROD/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA_/PROD/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", False, []],
    # Ci-dessous, on aimerait plutôt que `PROD__` ne soit pas pris pour le nom du segment. Mais aucune vérification n'est faite pour l'instant.
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/PROD__/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", False, []],
    ["/Users/louis/LV", False, []],
    ["00-Q0", False, []],
    ["", False, []],
    # ["/Volumes/Public/TRANSFORMATIONS/25-Q2/FEMME/MON-MNG/TRA/MARO/25-Q2-MON-MNG-ON-THE-GO-GM-MNG-(014)-241011.ai", False, []],
    # ["/Volumes/public/TRANSFORMATIONS/23-Q1/HOMME/LADY-B-M/INFINITY-DOTS/LIV/BELT/23-Q1-LVxYK-M-INFINITY-DOTS-BELT-40-(03)-220802.dxf", True, []],
    ["/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG/ARCHIVES/abc/24-Q1-SHOW-M-SS24-DAMIER-POP-PYRAMIDE-(04)-250206.ai", False, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG/ARCHIVES/24-Q1-SHOW-M-SS24-DAMIER-POP-PYRAMIDE-(03)-250206.ai", False, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG", False, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG/24-Q1-SHOW-M-SS24-DAMIER-POP-MODELE-(01)-250206.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/25-Q4/FEMME/MNG-SEQUINS/TRA/SLG/25Q4-MNG-SEQUINS-POCHETTE-ACCESSOIRES-(02)-250227.ai", False, []], # Manque un tiret dans le trimestre.
    ["/Volumes/Public/TRANSFORMATIONS/25-Q4/FEMME/X-MAS/WINTERY-TRAVEL/TRA/SLG/25-Q4-X-MAS-WINTERY-TRAVEL-PFLISAaaaa-(03)-250129.ai", False, []], # Minuscules dans le nom du produit.
    ["/Volumes/Public/TRANSFORMATIONS/25-Q4/FEMME/X-MAS/WINTERY-TRAVEL/TRA/SLG/25-Q4-X-MAS-WINTERY-TRAVEL-PFLISAAAA-(03)-250129.ai", True, []],
    ["/Volumes/public/TRANSFORMATIONS/PLACEMENT/26-Q1/FEMME/CLASH-SS/BLUE-FLOWER-IKAT/TRA/DUPLI/MOTIF/26-Q1-CLASH-SS-BLUE-FLOWER-IKAT-MOTIF-SMALL-500X500-(02)-250320.ai", True, []], # Valide mais le `X` devrait être minuscule : ici le nom du produit identifié est `"MOTIF-SMALL-500X500`.
    ["/Volumes/public/TRANSFORMATIONS/PLACEMENT/26-Q1/FEMME/CLASH-SS/BLUE-FLOWER-IKAT/TRA/DUPLI/MOTIF/26-Q1-CLASH-SS-BLUE-FLOWER-IKAT-MOTIF-SMALL-500x500-(02)-250320.ai", True, []], # Ajout des éventuelles dimensions du produit.
]

for path, expected_return, _ in test_paths:
    path_infos, path_is_valid, error_messages = parse_path(path)
    if path_is_valid != expected_return:
        print(path_infos)
        print(path)
        if error_messages:
            for error_message in error_messages:
                print(f"- {error_message}")

In [6]:
import requests
import json

url = 'https://product-library.vuitton.net/api/search/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:136.0) Gecko/20100101 Firefox/136.0',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'fr,fr-FR;q=0.8,en-US;q=0.5,en;q=0.3',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Content-Type': 'application/json',
    'Origin': 'https://product-library.vuitton.net',
    'Connection': 'keep-alive',
    'Referer': 'https://product-library.vuitton.net/',
    'Cookie': 'tokenId=1e065749-d0e0-4caf-a0a6-87e2777ff102',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'Priority': 'u=0',
    'TE': 'trailers'
}

payload = {
    "locale": "en_US",
    "size": 2,
    "from": 0,
    "search": "",
    "currency": "USD",
    "sortBy": {
        "field": "relevance",
        "isAsc": False
    },
    "filters": [
        {
            "field": "excludeObsoletes",
            "value": False,
            "key": "exclude_obsoletes_label"
        },
        {
            "field": "includeCanceled",
            "value": False,
            "key": "include_cancelled_label"
        },
        {
            "field": "includeSpecialCmd",
            "value": False,
            "key": "include_special_order_label"
        },
        {
            "field": "includeNonSellable",
            "value": False,
            "key": "include_non_sellable_label"
        },
        {
            "field": "gender",
            "value": "Men"
        }
    ]
}

response = requests.post(url, headers=headers, json=payload)

# Afficher le résultat
print(response.status_code)
print(json.dumps(response.json(), indent=4))

200
{
    "nbHits": 69742,
    "products": [
        {
            "_document_id": "d1d82da54d7b5eecf3aee9f6844b83a9",
            "_score": 5.6410007,
            "idProduct": "nvprod3410174v",
            "skuCode": "1A9ULK",
            "price": 4000,
            "productName": " 3-Piece Jacket",
            "sapName": "AF 3 PIECES JACKET",
            "sapModel": null,
            "material": null,
            "materialId": null,
            "launchDate": "20-01-2022",
            "retrievalDate": "30-10-2022",
            "images": [
                "https://louisvuitton.com/images/is/image/lv/1/PP_VP_L/louisvuitton--HMJ62EJ91325_PM2_Front view.jpg",
                "https://louisvuitton.com/images/is/image/lv/1/PP_VP_L/louisvuitton--HMJ62EJ91325_PM1_Worn view.jpg",
                "https://louisvuitton.com/images/is/image/lv/1/PP_VP_L/louisvuitton--HMJ62EJ91325_PM1_Closeup view.jpg",
                "https://louisvuitton.com/images/is/image/lv/1/PP_VP_L/louisvuitton--HMJ62EJ91325

In [34]:
# !pip install fast-mail-parser

import re

from fast_mail_parser import parse_email, ParseError

# EMAIL_PATH = "/Volumes/Public/TRANSFORMATIONS/ARTYCAP/EDITION_4/KY-ROUILLE/REF/RE- Fwd- Kick Off ARTCAPUCINES KY.eml'"
EMAIL_PATH = "/Volumes/Design_Numerique/PLACEMENT/23-Q4/FEMME/SHOW-W-FW23/MONOPANAME-CUIR/REF/Celine/Fwd_ PETITE MALLE PLAQUE DE RUE SPECIALE COREE.eml"
with open(EMAIL_PATH, 'r') as f:
    message_payload = f.read()

try:
    email = parse_email(message_payload)
except ParseError as e:
    print("Failed to parse email: ", e)
    sys.exit(1)

# print(email.subject)
# print(email.date)
# print(email.text_plain[0])
texte = email.text_plain[0]
texte = re.sub(
    r"(\S+\.(pdf|ai|psd))<https://urldefense\.com[^>]*>", r"<Pièce jointe \1>", texte
)
texte = re.sub(r"<https://urldefense\.com[^>]*>", "", texte)
text = re.sub(r'<mailto:[^>]+>', '', texte)
print(text)
# print(email.text_html)
# print(email.headers)

# for attachment in email.attachments:
#     print(attachment.mimetype)
#     # print(attachment.content)
#     print(attachment.filename)



Envoyé à partir de Outlook pour iOS<https://aka.ms/o0ukef>
________________________________
De : Celine MORTIER BJORNLUND <celine.mortierbjornlund@louisvuitton.com>
Envoyé : Friday, March 17, 2023 2:17:11 PM
À : Carole CHAUVEAU <carole.chauveau@louisvuitton.com>; Axel RAIWET <axel.raiwet.ext@louisvuitton.com>; France JACQUES <france.jacques@louisvuitton.com>; Marilyne MILLOT <marilyne.millot@louisvuitton.com>; Tom BODOUX <tom.bodoux@louisvuitton.com>
Cc : Barbara DANIEL TOLZA <barbara.danieltolza@louisvuitton.com>; Eleonore BONHOMME DE ROQUEFEUIL <eleonore.bonhommederoquefeuil@louisvuitton.com>; Laure Darcy <laure.darcy@pelletteriepalladio.it>; Pauline TATLOT JEAN CHARLES <pauline.tatlotjeancharles@louisvuitton.com>
Objet : Re: PETITE MALLE PLAQUE DE RUE SPECIALE COREE


Bonjour,



On lance bien les fichiers réalisé par Axel mardi, je les remet en PJ.

Merci

Céline



De : Carole CHAUVEAU <carole.chauveau@louisvuitton.com>
Date : vendredi, 17 mars 2023 à 13:19
À : Axel RAIWET <axel

In [12]:
# !pip install uform

# Permet de récupérer un modèle malgré le pare-feu pourri de LV.
import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

import configparser
from uform import get_model, Modality

# processors, models = get_model('unum-cloud/uform3-image-text-english-small')
processors, models = get_model('unum-cloud/uform3-image-text-multilingual-base')

model_text = models[Modality.TEXT_ENCODER]
model_image = models[Modality.IMAGE_ENCODER]
processor_text = processors[Modality.TEXT_ENCODER]
processor_image = processors[Modality.IMAGE_ENCODER]

/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/late

In [14]:
import requests
from io import BytesIO
from PIL import Image

image_url = 'https://media-cdn.tripadvisor.com/media/photo-s/1b/28/6b/53/lovely-armenia.jpg'
image = Image.open(BytesIO(requests.get(image_url).content))
image_data = processor_image(image)
image_features, image_embedding = model_image.encode(image_data, return_features=True)

# text = 'a cityscape bathed in the warm glow of the sun, with varied architecture and a towering, snow-capped mountain rising majestically in the background'
# text_data = processor_text(text)
# text_features, text_embedding = model_text.encode(text_data, return_features=True)
# print(text_features)

SSLError: HTTPSConnectionPool(host='media-cdn.tripadvisor.com', port=443): Max retries exceeded with url: /media/photo-s/1b/28/6b/53/lovely-armenia.jpg (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1017)')))

In [16]:
import os
from PIL import Image

# Chemin du dossier contenant les images
IMAGE_FOLDER = "/Users/gavaldalo/Vuitton/placement-git/app-git/media/"
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.gif', '.bmp'}


# Parcours récursif du dossier avec scandir
def process_images(folder_path):
    embeddings = {}

    def scan_directory(path):
        with os.scandir(path) as entries:
            for entry in entries:
                if entry.is_file() and os.path.splitext(entry.name)[1].lower() in IMAGE_EXTENSIONS:
                    try:
                        # Ouverture et prétraitement de l'image
                        image = Image.open(entry.path)
                        image_data = processor_image(image)

                        # Génération de l'embedding
                        _, image_embedding = model_image.encode(image_data, return_features=True)
                        print(image_embedding.shape)

                        # Stockage de l'embedding avec le chemin relatif comme clé
                        rel_path = os.path.relpath(entry.path, folder_path)
                        embeddings[rel_path] = image_embedding

                    except Exception as e:
                        print(f"Erreur lors du traitement de {entry.path}: {str(e)}")

                elif entry.is_dir():
                    scan_directory(entry.path)

    scan_directory(folder_path)
    return embeddings


# Traitement des images
embeddings_dict = process_images(IMAGE_FOLDER)


(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)
(1, 256)


KeyboardInterrupt: 

In [5]:
from multilingual_clip import pt_multilingual_clip
import transformers

# Permet de récupérer un modèle malgré le pare-feu pourri de LV.
import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

texts = [
    'Three blind horses listening to Mozart.',
    'Älgen är skogens konung!',
    'Wie leben Eisbären in der Antarktis?',
    'Вы знали, что все белые медведи левши?'
]
model_name = 'M-CLIP/XLM-Roberta-Large-Vit-L-14'

# Load Model & Tokenizer
model = pt_multilingual_clip.MultilingualCLIP.from_pretrained(model_name)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

embeddings = model.forward(texts, tokenizer)
print(embeddings.shape)

/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/late

torch.Size([4, 768])


/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/gavaldalo/Vuitton/placement-git/app-git/venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/late

In [7]:
# !pip install multilingual-clip torch
!pip install onnxruntime --upgrade

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.6/33.6 MB 11.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import pdftotext

# Ouvrir le fichier PDF
with open('/Users/gavaldalo/SHOW SS26 - MODELARIO - DUPLIS 1.pdf', 'rb') as file:
    # Charger le PDF
    pdf = pdftotext.PDF(file)

    # Extraire le texte des 3 premières pages
    for i in range(len(pdf)):
        print(f"=== Page {i + 1} ===")
        print(pdf[i])
        print()


=== Page 1 ===
EARL GREY

LOUIS VUITTON - STUDIO HOMME

SHOW SS26 - MODELARIO - WES ANDERSON PRINT

19. MAI

9

1.

2.

NIL GM

NIL MM

KEEPALL 35

KEEPALL 25

SAC PLAT MM
SANS BDC

SAC PLAT PM
SANS BDC

CHRISTOPHER MESSANGER
TBC

SPEEDY BUTTERSOFT
EMBROIDERY
DENIM BASE AS RTW

3.
CHRISTOPHER MM

MATERIALS:
MAIN: DRIFT OR SIENNA
TRIM: DRIFT OR SIENNA
LINING: SUEDE
PMET: OR BRUT

TRAVEL

WEBBING: CHARM: TBC WHISTLE
NUMBERS : HANDPAINTED ON FINAL SELECTION

KEEPALL 50

SIRIUS 50

1



=== Page 2 ===
LOUIS VUITTON - STUDIO HOMME

SHOW SS26 - MODELARIO - WES ANDERSON PRINT

EARL GREY

19. MAI

2



=== Page 3 ===
EARL GREY ON DAMIER
CARAMEL TRIM

LOUIS VUITTON - STUDIO HOMME

SHOW SS26 - MODELARIO - WES ANDERSON ON DAMIER

19. MAI

9

1.

2.

SPEEDY 30

SPEEDY 25

SAC PLAT MM

NIL GM

NIL MM

CHRISTOPHER MM

SAC PLAT PM

3.

MATERIALS:
MAIN: ON DAMIER EBENE HERITAGE
TRIM: CHOCOLAT OR VVN CARAMELO SFUMATO
LINING: COCO 317
PMET: OR BRUT

BOOK CLUTCH
(PRINT INSIDE TBC)
TRAVEL

WEBBING: CHAR